# EduAssist Fine-Tuning — Google Colab
Education & Training industry LLM chatbot.

## 1. Install dependencies

In [ ]:
!pip -q install -U transformers datasets peft accelerate bitsandbytes trl sentencepiece


In [ ]:
import json, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer


## 2. Upload dataset
Upload `eduassist_dataset.jsonl` to Colab, then set the path below.

In [ ]:
DATA_PATH = '/content/eduassist_dataset.jsonl'
MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'  # Replace with another compatible instruction model if needed
dataset = load_dataset('json', data_files=DATA_PATH, split='train')
dataset = dataset.train_test_split(test_size=0.1, seed=42)
dataset


## 3. Format examples

In [ ]:
def format_example(x):
    return f"### Instruction:\n{x['instruction']}\n\n### Response:\n{x['response']}"
train_ds = dataset['train']
test_ds = dataset['test']


## 4. Load tokenizer/model
The exact model and quantization settings may need adjustment depending on current Colab/Hugging Face compatibility.

In [ ]:
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb, device_map='auto')
model = prepare_model_for_kbit_training(model)


## 5. LoRA configuration

In [ ]:
peft_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM', target_modules=['q_proj','k_proj','v_proj','o_proj'])


## 6. Fine-tune
Start with 3–5 epochs. The assignment allows up to 25 epochs; more is not automatically better.

In [ ]:
args = TrainingArguments(
    output_dir='/content/eduassist-output',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy='epoch',
    report_to='none',
    fp16=True,
)
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    peft_config=peft_config,
    formatting_func=format_example,
    args=args,
)
trainer.train()
trainer.save_model('/content/eduassist-adapter')
tokenizer.save_pretrained('/content/eduassist-adapter')


## 7. Manual evaluation
Use held-out questions and record relevance, correctness, and out-of-domain behavior. Do not report invented metrics.

In [ ]:
for item in test_ds.select(range(min(10, len(test_ds)))):
    print('Q:', item['instruction'])
    print('Expected:', item['response'])
    print('-'*80)


## 8. Next steps
Export the adapter/model, connect it to the Gradio application, and evaluate against a larger unseen test set.